# 04 — Multi-curve bootstrap: one global solve, not two sequential ones

## The two questions every post-LIBOR price asks

A cashflow at $t$ is discounted with $P_d(0,t)$ from the **discount curve** (OIS on the collateral rate: SOFR for a
USD CSA). A floating coupon on index $I$ over $[t_s, t_e]$ is projected off the **projection curve** of that index:

$$R_I(t_s,t_e) \;=\; \frac{1}{\tau}\left(\frac{P_I(0,t_s)}{P_I(0,t_e)} - 1\right)$$

for a compounded overnight index ($\tau$ in ACT/360). For SOFR the two curves coincide; for Fed Funds they do not,
and the FF curve is built from FF–SOFR basis swaps. A par OIS swap with fixed rate $K$ satisfies

$$K \sum_i \tau_i P_d(t_i) \;=\; \sum_j \left(R_j + s\right)\tau_j P_d(t_j^{pay}),$$

so with $N$ instruments quoted at $q$ and a curve set described by node ordinates $z$ (we use $y = -\log P$ at
the pillars), the bootstrap is the square system

$$F(z;\,q) = 0, \qquad F_i = \text{model}_i(z) - q_i .$$

## Why "bootstrap" in the textbook sense fails

If every instrument on curve A depends only on curve A's nodes and curve B's instruments depend on A and B, you
can solve A first and then B — sequentially, node by node. That stops being true the moment an instrument on
curve **A** depends on curve **B**. In the USD set here the 25Y and 30Y SOFR pillars are pinned by FF–SOFR basis
swaps (there is no SOFR OIS quote there in this file), whose FF leg needs the FF curve, whose own instruments
discount off SOFR. Neither curve can be finished before the other: $\partial F_i/\partial z_j \neq 0$ across
the block boundary, and a one-pass sequential solve leaves a residual on exactly those instruments. The fix is
Newton on the whole $z$ at once, with the residuals scaled to basis points so the 30y does not drown the 1-week.

## Interpolation and the forward curve

Nodes only pin $P$ at the pillars; the instantaneous forward $f(t) = -\partial_t \log P$ between pillars is the
interpolator's business (notebook 02). Linear on $y$ gives piecewise-flat forwards with jumps at every pillar —
which is what the OIS front end looks like anyway (policy meetings are steps). Monotone convex gives continuous
forwards; both are local, so a bump to the 10y quote does not move the 2y forward.

## A hand example: two OIS quotes, two nodes

Single-period 1Y OIS at $K_1 = 3.48\%$ with $\tau_1 = 1.0139$ (ACT/360) gives

$$P_1 = \frac{1}{1 + K_1 \tau_1}.$$

The 2Y OIS at $K_2 = 3.32\%$ pays annually, so with the telescoped float leg $1 - P_2$:

$$K_2(\tau_1 P_1 + \tau_2 P_2) = 1 - P_2 \quad\Rightarrow\quad P_2 = \frac{1 - K_2 \tau_1 P_1}{1 + K_2 \tau_2}.$$

That *is* a sequential bootstrap — it works here because the 2Y depends on the 1Y but not the other way round.
Check the numbers by hand, then see what the C++ does with 37 instruments and two coupled curves.

In [ ]:
import _util
from _util import run_demo
import numpy as np, matplotlib.pyplot as plt

K1, K2, tau1, tau2 = 0.0348, 0.0332, 365/360, 366/360
P1 = 1 / (1 + K1 * tau1)
P2 = (1 - K2 * tau1 * P1) / (1 + K2 * tau2)
print(f"P(1Y) = {P1:.6f}   zero = {-np.log(P1)/1.0:.5%}")
print(f"P(2Y) = {P2:.6f}   zero = {-np.log(P2)/2.0:.5%}")
# Reprice check: the 2Y swap at K2 must have zero PV.
print("2Y PV per unit notional:", K2 * (tau1 * P1 + tau2 * P2) - (1 - P2))

## The quote file, validated in Python first

`ratelib.market.quotes` runs the desk's sanity checks (units, IMM strip, duplicate pillars, cross-assignment).
This cell works without any C++.

In [ ]:
from ratelib.market import load_quotes, validate_quotes
from ratelib.market.quotes import summary
from ratelib.demos import repo_root
import datetime as dt

qs = load_quotes(repo_root() / "data" / "quotes" / "usd_2026-09-04.json", validate=False)
print(summary(qs))
for w in validate_quotes(qs, today=dt.date(2026, 9, 4)):
    print("warning:", w)

## The global solve

`rl_demo_bootstrap reprice` builds every instrument, solves all 37 nodes of both curves at once and prints the
repricing report.

In [ ]:
from ratelib.market import CurveReport, load_reprice_csv
rep = load_reprice_csv(run_demo("rl_demo_bootstrap", "reprice", "linear").to_csv(index=False))
print(rep.verdict())
rep.by_curve()

In [ ]:
rep.table[["id", "type", "curve", "maturity", "quote", "model", "residual_bp"]]

## Forwards under the two interpolators

Piecewise-flat forwards (linear on log-DF) versus continuous forwards (monotone convex). Look at the futures
region: the IMM pillars every three months make the flat-forward steps visible, and at the turn of the year
there is nothing yet: turn quotes are excluded by default (`InstrumentSetOptions::includeTurns`), so the year-end spike is absent here and appears in the turn test once the turn residual is written.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for interp, lab in [("linear", "linear on log-DF"), ("mc", "monotone convex")]:
    fw = run_demo("rl_demo_bootstrap", "forwards", interp, 6, 3)
    ax.plot(fw["t"], 100 * fw["sofr_fwd"], label=f"SOFR o/n fwd, {lab}")
    ax.plot(fw["t"], 100 * fw["ff_fwd"], "--", label=f"FF o/n fwd, {lab}", alpha=0.7)
ax.set_xlabel("years from as-of"); ax.set_ylabel("overnight forward, %"); ax.legend(); ax.set_title("Bootstrapped overnight forwards")
plt.show()

## Sequential versus global

One pass of the curve-by-curve bootstrap (`solveSequential`) reprices every SOFR OIS and future — they do not
see the FF curve — but the 25Y/30Y basis swaps assigned to the SOFR curve were solved against the *guessed*
FF curve and keep a residual of several basis points. The global Newton removes it.

In [ ]:
sq = run_demo("rl_demo_bootstrap", "sequential", "linear")
fig, ax = plt.subplots(figsize=(11, 4))
x = np.arange(len(sq))
ax.bar(x - 0.2, sq["residual_bp_sequential"].abs(), 0.4, label="after one sequential pass")
ax.bar(x + 0.2, sq["residual_bp_global"].abs(), 0.4, label="after the global solve")
ax.set_yscale("log"); ax.set_xticks(x); ax.set_xticklabels(sq["id"], rotation=90, fontsize=7)
ax.set_ylabel("|residual|, bp"); ax.legend(); ax.set_title("Why the solve is global")
plt.tight_layout(); plt.show()
print(sq.sort_values("residual_bp_sequential", key=abs, ascending=False).head(5))

## Snapshot hash

Every result downstream carries this number. Same quotes, same fixings, same code → same hash on every
machine.

In [ ]:
print(run_demo("rl_demo_bootstrap", "hash", "linear").columns[0])